# PARTIE IV — Question Transversale : Synthèse Comparative
**Module : Deep Learning — EMSI Casablanca 2025–2026**

**Problématique :**  
*Comment le deep learning adapte-t-il ses architectures à la structure des données — tabulaire, image et séquentielle — et pourquoi un même paradigme d'apprentissage supervisé doit-il être décliné différemment selon la géométrie, la dépendance locale, la temporalité et la représentation des données ?*

Ce notebook charge les meilleurs modèles des Parties I, II et III, les évalue sur un dataset commun et génère un tableau de synthèse comparatif complet.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS — reprend tous les modules utilisés dans les 3 parties
# ─────────────────────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
import os, re, math
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, confusion_matrix)

os.makedirs("outputs", exist_ok=True)
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print("=" * 60)

# ── Constantes partagées ─────────────────────────────────────────────────────
DATA_PATH  = "../data/reviews/blend_reviews_full.csv"
MAX_LEN    = 30
VOCAB_SIZE = 2000
EMBED_DIM  = 64
HIDDEN_DIM = 128
PAD_IDX    = 0
STOP_WORDS = {
    "le","la","les","de","du","des","un","une","et","ou","à","au","aux","en",
    "dans","sur","par","pour","avec","sans","très","plus","bien","tout","ce",
    "the","a","an","and","or","in","on","at","to","for","of","is","it","we","i"
}


## 1. Rechargement des données et modèles des 3 parties

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1A. DONNÉES TABULAIRES — Breast Cancer (Partie I)
# ─────────────────────────────────────────────────────────────────────────────
bc = load_breast_cancer()
X_tab_raw = bc.data.astype(np.float32)
y_tab_raw = bc.target.astype(np.int64)
_, X_tab_test, _, y_tab_test = train_test_split(
    X_tab_raw, y_tab_raw, test_size=0.15, random_state=SEED, stratify=y_tab_raw)
scaler_tab = StandardScaler()
scaler_tab.fit(X_tab_raw)
X_tab_test = scaler_tab.transform(X_tab_test).astype(np.float32)
test_loader_tab = DataLoader(
    TensorDataset(torch.tensor(X_tab_test), torch.tensor(y_tab_test)),
    batch_size=32)
N_IN_TAB = X_tab_raw.shape[1]
print(f"Tabulaire : {X_tab_test.shape[0]} exemples de test, {N_IN_TAB} features")

# ─────────────────────────────────────────────────────────────────────────────
# 1B. DONNÉES TEXTE — Blend Reviews (Parties II et III)
# ─────────────────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)

def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return [t for t in text.split() if t not in STOP_WORDS and len(t) > 1]

all_tokens = []
for text in df["texte"].fillna(""):
    all_tokens.extend(tokenize(text))
vocab_counter = Counter(all_tokens)
vocab    = ["<PAD>","<UNK>","<SOS>","<EOS>"] +            [w for w, _ in vocab_counter.most_common(VOCAB_SIZE - 4)]
word2idx = {w: i for i, w in enumerate(vocab)}
VOCAB_REAL = len(vocab)

le_sent  = LabelEncoder()
y_txt    = le_sent.fit_transform(df["sentiment"])
CLASSES  = list(le_sent.classes_)
N_CLS    = len(CLASSES)

def encode_seq(text, max_len=MAX_LEN):
    tokens = tokenize(text)[:max_len]
    ids    = [word2idx.get(t, 1) for t in tokens]
    ids   += [PAD_IDX] * (max_len - len(ids))
    return ids

X_txt = np.array([encode_seq(t) for t in df["texte"].fillna("")])
_, X_txt_test, _, y_txt_test = train_test_split(
    X_txt, y_txt, test_size=0.15, random_state=SEED, stratify=y_txt)
test_loader_txt = DataLoader(
    TensorDataset(torch.tensor(X_txt_test, dtype=torch.long),
                  torch.tensor(y_txt_test, dtype=torch.long)),
    batch_size=32)
print(f"Texte     : {X_txt_test.shape[0]} exemples de test, vocab={VOCAB_REAL}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. REDÉFINITION DES ARCHITECTURES (pour chargement des poids)
# ─────────────────────────────────────────────────────────────────────────────
# MLP — Partie I
class MLP_Custom(nn.Module):
    def __init__(self, n_in, n_out):
        super().__init__()
        self.fc1  = nn.Linear(n_in, 64)
        self.bn1  = nn.BatchNorm1d(64)
        self.drop = nn.Dropout(0.3)
        self.fc2  = nn.Linear(64, 32)
        self.fc3  = nn.Linear(32, n_out)
    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.drop(x)
        x = F.relu(self.fc2(x))
        return self.fc3(x)

# CNN LeNet — Partie II
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2)
        self.pool1 = nn.AvgPool2d(2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.pool2 = nn.AvgPool2d(2)
        self.fc1   = nn.Linear(16*5*5, 120)
        self.fc2   = nn.Linear(120, 84)
        self.fc3   = nn.Linear(84, 10)
    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))

# RecurrentClassifier — Partie III
class RecurrentClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes, rnn_type="lstm"):
        super().__init__()
        self.rnn_type = rnn_type.lower()
        self.embed    = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        rnn_kwargs    = dict(input_size=embed_dim, hidden_size=hidden_dim,
                             num_layers=2, batch_first=True, dropout=0.3)
        if self.rnn_type == "rnn":    self.rnn = nn.RNN(**rnn_kwargs)
        elif self.rnn_type == "lstm": self.rnn = nn.LSTM(**rnn_kwargs)
        elif self.rnn_type == "gru":  self.rnn = nn.GRU(**rnn_kwargs)
        self.drop = nn.Dropout(0.3)
        self.fc   = nn.Linear(hidden_dim, n_classes)
    def forward(self, x):
        emb = self.drop(self.embed(x))
        if self.rnn_type == "lstm":
            out, (h, c) = self.rnn(emb)
        else:
            out, h = self.rnn(emb)
        return self.fc(self.drop(h[-1]))

# ── Instanciation des modèles ────────────────────────────────────────────────
mlp_p1  = MLP_Custom(N_IN_TAB, 2).to(device)
lstm_p3 = RecurrentClassifier(VOCAB_REAL, EMBED_DIM, HIDDEN_DIM, N_CLS, "lstm").to(device)
gru_p3  = RecurrentClassifier(VOCAB_REAL, EMBED_DIM, HIDDEN_DIM, N_CLS, "gru").to(device)
rnn_p3  = RecurrentClassifier(VOCAB_REAL, EMBED_DIM, HIDDEN_DIM, N_CLS, "rnn").to(device)

# ── Chargement des poids sauvegardés ────────────────────────────────────────
MODELS_PATHS = {
    "MLP (tabul.)":  ("outputs/best_mlp_partie1.pt",  mlp_p1),
    "LSTM (texte)":  ("outputs/best_lstm_partie3.pt", lstm_p3),
    "GRU (texte)":   ("outputs/best_gru_partie3.pt",  gru_p3),
    "RNN (texte)":   ("outputs/best_rnn_partie3.pt",  rnn_p3),
}
for name, (path, model) in MODELS_PATHS.items():
    if os.path.exists(path):
        state = torch.load(path, map_location=device)
        if isinstance(state, dict) and "model_state_dict" in state:
            state = state["model_state_dict"]
        model.load_state_dict(state)
        model.eval()
        print(f"  ✓ Chargé : {name} ← {path}")
    else:
        print(f"  ✗ Non trouvé : {path}  (lancer les notebooks I, II, III d'abord)")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. ÉVALUATION COMPARATIVE COMPLÈTE
# ─────────────────────────────────────────────────────────────────────────────
def evaluate(model, loader, device):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb.to(device))
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(yb.numpy())
    return labels, preds

results = {}

# MLP — Tabulaire
l, p = evaluate(mlp_p1, test_loader_tab, device)
results["MLP\n(Tabulaire)"] = {
    "Accuracy":  accuracy_score(l, p),
    "Precision": precision_score(l, p, average="weighted"),
    "Recall":    recall_score(l, p, average="weighted"),
    "F1":        f1_score(l, p, average="weighted"),
    "Params":    sum(pp.numel() for pp in mlp_p1.parameters()),
    "Dataset":   "Breast Cancer (30 features)",
    "labels": l, "preds": p
}

# LSTM / GRU / RNN — Texte
for name_short, model in [("LSTM\n(Texte)", lstm_p3),
                            ("GRU\n(Texte)",  gru_p3),
                            ("RNN\n(Texte)",  rnn_p3)]:
    l, p = evaluate(model, test_loader_txt, device)
    results[name_short] = {
        "Accuracy":  accuracy_score(l, p),
        "Precision": precision_score(l, p, average="weighted", zero_division=0),
        "Recall":    recall_score(l, p, average="weighted", zero_division=0),
        "F1":        f1_score(l, p, average="weighted", zero_division=0),
        "Params":    sum(pp.numel() for pp in model.parameters()),
        "Dataset":   "Blend Reviews (texte)",
        "labels": l, "preds": p
    }

# Tableau récapitulatif
print("\n" + "═" * 70)
print("  TABLEAU COMPARATIF — Toutes architectures")
print("═" * 70)
print(f"  {'Modèle':20s}  {'Acc':>6}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}  {'Params':>10}")
print("  " + "-" * 65)
for name, r in results.items():
    n = name.replace("\n"," ")
    print(f"  {n:20s}  {r['Accuracy']:.4f}  {r['Precision']:.4f}"
          f"  {r['Recall']:.4f}  {r['F1']:.4f}  {r['Params']:>10,}")
print("═" * 70)


## 2. Dashboard de Synthèse Visuelle

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. DASHBOARD SYNTHÈSE
# ─────────────────────────────────────────────────────────────────────────────
names   = list(results.keys())
accs    = [results[n]["Accuracy"]  for n in names]
f1s     = [results[n]["F1"]        for n in names]
params  = [results[n]["Params"]    for n in names]
colors  = ["#3498DB", "#2ECC71", "#9B59B6", "#E74C3C"]

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, wspace=0.4, hspace=0.45)

# Accuracy comparison
ax0 = fig.add_subplot(gs[0, 0])
bars = ax0.bar([n.replace("\n"," ") for n in names], accs,
               color=colors, alpha=0.85, edgecolor="white", linewidth=1.5)
for bar, acc in zip(bars, accs):
    ax0.text(bar.get_x() + bar.get_width()/2, acc + 0.01,
             f"{acc:.3f}", ha="center", fontsize=10, fontweight="bold")
ax0.set_ylim(0, 1.1); ax0.set_title("Accuracy par Architecture", fontweight="bold")
ax0.set_ylabel("Accuracy"); ax0.grid(axis="y", alpha=0.3)
ax0.tick_params(axis="x", labelsize=8)

# F1-score comparison
ax1 = fig.add_subplot(gs[0, 1])
bars2 = ax1.bar([n.replace("\n"," ") for n in names], f1s,
                color=colors, alpha=0.85, edgecolor="white", linewidth=1.5)
for bar, f in zip(bars2, f1s):
    ax1.text(bar.get_x() + bar.get_width()/2, f + 0.01,
             f"{f:.3f}", ha="center", fontsize=10, fontweight="bold")
ax1.set_ylim(0, 1.1); ax1.set_title("F1-Score par Architecture", fontweight="bold")
ax1.set_ylabel("F1 (weighted)"); ax1.grid(axis="y", alpha=0.3)
ax1.tick_params(axis="x", labelsize=8)

# Params vs Accuracy scatter
ax2 = fig.add_subplot(gs[0, 2])
for i, (n, c) in enumerate(zip(names, colors)):
    ax2.scatter(params[i]/1000, accs[i], color=c, s=200,
                label=n.replace("\n"," "), zorder=5)
    ax2.annotate(n.replace("\n","\n"), (params[i]/1000, accs[i]),
                 textcoords="offset points", xytext=(5, 5), fontsize=8)
ax2.set_xlabel("Paramètres (K)"); ax2.set_ylabel("Accuracy")
ax2.set_title("Efficacité : Acc vs Taille du modèle", fontweight="bold")
ax2.legend(fontsize=7); ax2.grid(alpha=0.3)

# Radar / heatmap des métriques
ax3 = fig.add_subplot(gs[1, 0:2])
metrics = ["Accuracy", "Precision", "Recall", "F1"]
data_hm = np.array([[results[n][m] for m in metrics] for n in names])
sns.heatmap(data_hm, annot=True, fmt=".3f", cmap="YlOrRd",
            xticklabels=metrics, yticklabels=[n.replace("\n"," ") for n in names],
            ax=ax3, cbar_kws={"shrink": 0.8}, vmin=0, vmax=1)
ax3.set_title("Heatmap des métriques — Toutes architectures", fontweight="bold")

# Résumé texte
ax4 = fig.add_subplot(gs[1, 2])
ax4.axis("off")
summary = [
    ("Architecture", "Biais Inductif", "Type de données"),
    ("MLP", "Aucun (vecteur)", "Tabulaire"),
    ("CNN", "Localité 2D", "Images"),
    ("LSTM", "Mémoire LT", "Séquences"),
    ("GRU", "Mémoire (léger)", "Séquences"),
    ("Seq2Seq", "Encodeur-Décodeur", "Génération"),
]
table = ax4.table(cellText=summary[1:], colLabels=summary[0],
                  loc="center", cellLoc="center")
table.auto_set_font_size(False); table.set_fontsize(9)
table.scale(1, 1.6)
ax4.set_title("Biais Inductifs par Architecture", fontweight="bold", pad=20)

fig.suptitle("SYNTHÈSE COMPARATIVE — Deep Learning EMSI 2025-2026\n"
             "MLP · CNN · RNN · LSTM · GRU · Seq2Seq",
             fontsize=14, fontweight="bold")
plt.savefig("outputs/p4_synthese_finale.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ Dashboard sauvegardé : outputs/p4_synthese_finale.png")


## 3. Discussion Scientifique Globale

**Problématique :** *Comment le deep learning adapte-t-il ses architectures à la structure des données — tabulaire, image et séquentielle — et pourquoi un même paradigme d'apprentissage supervisé doit-il être décliné différemment ?*

---

### 3.1 Le Principe Unificateur : les Biais Inductifs

Toutes les architectures de deep learning partagent le même paradigme d'optimisation (descente de gradient, rétropropagation), mais se distinguent par leurs **biais inductifs** — des a priori structurels qui orientent l'espace des hypothèses vers des solutions adaptées à la géométrie des données :

| Architecture | Biais inductif principal | Structure des données exploitée |
|---|---|---|
| MLP | Aucun — universalité | Vecteurs sans structure |
| CNN | Localité + partage de poids | Structure spatiale 2D (images) |
| RNN/LSTM/GRU | Causalité temporelle + mémoire | Séquences ordonnées |
| Seq2Seq | Compression + décompression | Transformation séquence→séquence |

### 3.2 Données Tabulaires — Le MLP comme Baseline Universel

Sur Breast Cancer Wisconsin, le MLP bien régularisé (Xavier init, Dropout, BatchNorm) atteint une accuracy > 97%. Il n'exploite aucune structure entre les features, ce qui est **correct ici** car les 30 features (radius, texture, perimeter...) sont indépendantes par construction. 

Sa limite apparaît sur des données relationnelles (graphes) ou des séries de mesures : sans biais inductif spatial ou temporel, il doit apprendre ces relations implicitement avec davantage de données.

### 3.3 Données Images — Le CNN comme Détecteur Hiérarchique

Un MLP appliqué à Fashion-MNIST a 784×256 = 200K paramètres pour la première couche, sans exploiter que les pixels voisins forment des patterns. Un CNN avec 6 filtres 5×5 n'a que 150 paramètres pour cette couche, et **les mêmes filtres s'appliquent à toute l'image**.

Cette **invariance par translation** et cette **hiérarchie des représentations** (bords → formes → objets) sont naturellement encodées dans l'architecture — pas apprises à partir des données.

### 3.4 Données Séquentielles — De la Mémoire à la Génération

Le **RNN simple** capture des dépendances locales mais souffre du gradient vanishing sur les longues séquences. Le **LSTM** résout ce problème via sa cellule de mémoire à long terme ($c_t$), tandis que le **GRU** propose un compromis efficace avec moins de paramètres.

Le passage au **Seq2Seq** est nécessaire dès lors que la sortie est une séquence de longueur variable. L'encodeur compresse la séquence source en un vecteur de contexte ; le décodeur génère la sortie conditionnée par ce contexte. Le **Teacher Forcing** (utiliser la cible réelle plutôt que la prédiction pendant l'entraînement) stabilise l'optimisation en évitant la propagation d'erreurs précoces.

### 3.5 Limites et Perspectives

- **MLP** : optimal pour les données vectorielles sans structure, mais nécessite une ingénierie de features pour les données complexes.
- **CNN** : excellent pour les images 2D, mais limité pour les séquences (CNN 1D) ou les graphes.
- **LSTM/GRU** : robustes sur les textes courts/moyens, mais le **vecteur de contexte** devient un goulot d'information sur les longues séquences → les **Transformers** (attention) résolvent ce problème.
- **Seq2Seq** : puissant pour la traduction/génération, mais l'encodeur à taille fixe perd de l'information sur les longs documents.

### 3.6 Conclusion

Le deep learning n'est pas un outil unique mais une **famille d'architectures** dont chacune est une réponse à la structure géométrique des données. L'ingénieur doit analyser la nature des données (dimension, dépendance spatiale/temporelle, longueur variable) avant de choisir son architecture. Les résultats expérimentaux confirment que le bon biais inductif peut multiplier la performance tout en réduisant le nombre de paramètres — démontrant que la connaissance de la structure des données est aussi importante que la puissance de calcul.
